# Contextual Bandits: from theory to a real Facebook Messenger experiment

**Persuasion at Scale, Class 22 (2026-04-15)**
**Eunji Kim and Chris Wiggins, Columbia University**

Companion notebook for Wednesday's lecture on contextual bandits. This runs in Google Colab with no setup: everything you need is loaded from public URLs.

Three acts:

1. **Warmup.** Classical 2-arm bandit, no context. Reminds you of Monday's material.
2. **LinUCB on a toy contextual bandit.** Synthetic 2-arm problem with 10-dimensional context vectors. We use per-arm ridge regression plus an upper confidence bonus, and plot how the algorithm's beliefs evolve.
3. **Real data.** We load the replication data from Offer-Westort, Rosenzweig and Athey (2024), "Battling the coronavirus 'infodemic' among social media users in Kenya and Nigeria," *Nature Human Behaviour*. 15,292 Facebook Messenger users, 40 treatment cells, real outcomes. We collapse the 40 arms to 4 interpretable options and run Linear Thompson Sampling on the real covariates.

You do not need to understand every line of code. The markdown cells and plots tell the story.

---

**Reading list for today**

- Offer-Westort, Rosenzweig and Athey (2024) - this notebook's data source
- Li, Chu, Langford and Schapire (2010) - the original LinUCB paper (Yahoo Today News)
- Liao, Greenewald, Klasnja and Murphy (2020) - Personalized HeartSteps

All three are linked in `assigned.md` for the lecture.

In [ ]:
# Setup. Only standard scientific Python.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

DATA_URL = (
    "https://raw.githubusercontent.com/gsbDBI/infodemic-replication/"
    "main/data/cleaned-data_2023-03-28.csv"
)
print("Ready.")

## Act 1: warmup - classical 2-arm bandit

Monday's material, in 40 lines of code.

Two ads. Ad A clicks with probability 0.12. Ad B clicks with probability 0.18. You do not know these rates. You have 500 chances to show an ad. Which ad do you pick each round?

Thompson Sampling maintains a Beta distribution over each arm's true click rate, samples a random "plausible world" each round, and picks the arm that wins in that sampled world. As evidence accumulates, the Betas tighten around the truth.

In [ ]:
# Act 1: classical Thompson Sampling on 2 arms.
true_rates = np.array([0.12, 0.18])   # Ad A, Ad B
T = 500

alpha = np.ones(2)   # Beta(1, 1) prior per arm
beta = np.ones(2)

picks = np.zeros(T, dtype=int)
clicks = np.zeros(T, dtype=int)
alpha_hist = np.zeros((T, 2))
beta_hist = np.zeros((T, 2))

rng = np.random.default_rng(42)
for t in range(T):
    samples = rng.beta(alpha, beta)          # draw one "plausible world"
    arm = int(np.argmax(samples))            # pick best arm in that world
    reward = int(rng.random() < true_rates[arm])
    if reward:
        alpha[arm] += 1
    else:
        beta[arm] += 1
    picks[t] = arm
    clicks[t] = reward
    alpha_hist[t] = alpha
    beta_hist[t] = beta

total_clicks = clicks.sum()
fraction_B = (picks == 1).cumsum() / np.arange(1, T + 1)

print(f"Total clicks after {T} rounds: {total_clicks}")
print(f"Ad B selected on {(picks == 1).sum()} of {T} rounds ({100 * (picks == 1).mean():.1f}%)")

In [ ]:
# Plot what happened in Act 1.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(np.arange(1, T + 1), fraction_B, color='#f59e0b', lw=2, label='P(pick Ad B)')
axes[0].axhline(1.0, color='grey', ls='--', alpha=0.5, label='target')
axes[0].set_xlabel('round t')
axes[0].set_ylabel('fraction picking Ad B')
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Thompson sampling homes in on Ad B')
axes[0].legend(loc='lower right')

rates_A = np.linspace(0, 0.4, 400)
post_A = np.array([
    np.exp((alpha_hist[t, 0] - 1) * np.log(rates_A + 1e-12) +
           (beta_hist[t, 0] - 1) * np.log(1 - rates_A + 1e-12))
    for t in [0, 50, 250, T - 1]
])
post_B = np.array([
    np.exp((alpha_hist[t, 1] - 1) * np.log(rates_A + 1e-12) +
           (beta_hist[t, 1] - 1) * np.log(1 - rates_A + 1e-12))
    for t in [0, 50, 250, T - 1]
])
post_A = post_A / post_A.max(axis=1, keepdims=True)
post_B = post_B / post_B.max(axis=1, keepdims=True)
shades_A = ['#c7d2fe', '#a5b4fc', '#818cf8', '#6366f1']
shades_B = ['#fde68a', '#fcd34d', '#fbbf24', '#f59e0b']
labels = ['t=1', 't=50', 't=250', f't={T}']
for i in range(4):
    axes[1].plot(rates_A, post_A[i], color=shades_A[i], lw=1.5, label=f'Ad A, {labels[i]}')
    axes[1].plot(rates_A, post_B[i], color=shades_B[i], lw=1.5, label=f'Ad B, {labels[i]}')
axes[1].axvline(true_rates[0], color='#6366f1', ls='--', alpha=0.5)
axes[1].axvline(true_rates[1], color='#f59e0b', ls='--', alpha=0.5)
axes[1].set_xlabel('click rate')
axes[1].set_ylabel('posterior density (normalized)')
axes[1].set_title('Beta posteriors sharpen over time')
axes[1].legend(loc='upper right', fontsize=7, ncol=2)

plt.tight_layout()
plt.show()
print("After 500 rounds the algorithm is nearly certain Ad B wins,")
print("and its Beta posterior for Ad B is concentrated near the true rate 0.18.")

## Act 2: LinUCB on a toy contextual bandit

Now the point of today's lecture. In classical bandits, every user looks the same. In contextual bandits, each user arrives with a **context vector** $x$, and the algorithm learns a function that maps context to expected reward for each arm.

**Setup.** Two arms. Each user has a 10-dimensional context vector (think: age, income, country, time of day, etc, normalized). The true expected reward for arm $a$ given context $x$ is $x^\top \beta_a$ for some unknown coefficient vectors $\beta_0$ and $\beta_1$.

**Algorithm: LinUCB.** At each round we fit ridge regression to our history so far (one regression per arm) to get an estimate $\hat\beta_a$. For an incoming context $x_t$, each arm's score is:

$$\text{score}_a(x_t) = x_t^\top \hat\beta_a + c \sqrt{x_t^\top A_a^{-1} x_t}$$

The first term is the predicted reward for that user on that arm. The second term is an uncertainty bonus: it gets big when we haven't seen many users like $x_t$ on arm $a$. We pick the arm with the highest score. High $c$ means more exploration, $c = 0$ is pure greedy.

This is the algorithm from Li, Chu, Langford & Schapire (2010), originally deployed on Yahoo Today News (Part 3's paper).

In [ ]:
# Before the simulation, look at what ONE user's context vector looks like.
# Each user is a 10-dimensional vector of "features" (normalized numbers around 0).
# Think of the features as things like: age, income, country, time of day, device type, etc.

rng_demo = np.random.default_rng(7)
one_user = rng_demo.standard_normal(10) * 0.5
print("One user's context vector (10 features):")
print(one_user.round(3))
print()
print("In the real world these would be interpretable columns in a database.")
print("Here they are just numbers. The algorithm does not care about meaning;")
print("it just learns which arm does well when the feature vector looks like this.")

In [ ]:
# Act 2: LinUCB on a 2-arm, 10-dim contextual bandit.

d = 10                                  # context dim
K = 2                                   # arms
T_ctx = 2000                            # horizon
c_ucb = 1.0                             # exploration constant
lam = 1.0                               # ridge penalty

rng2 = np.random.default_rng(7)

# True per-arm coefficient vectors. We pick them so neither arm dominates:
# arm 0 is better for contexts with positive first feature, arm 1 for negative.
beta_true = np.stack([
    np.array([ 0.9, 0.1, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1]),
    np.array([-0.8, 0.0, 0.2, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1]),
])

# Ridge regression state per arm: A (d x d) and b (d,).
A = np.stack([lam * np.eye(d) for _ in range(K)])
b = np.zeros((K, d))

# Logging.
rewards = np.zeros(T_ctx)
regrets = np.zeros(T_ctx)
beta_hat_hist = np.zeros((T_ctx, K, d))
arm_picks = np.zeros(T_ctx, dtype=int)

def linucb_score(x, A_a, b_a, c):
    A_inv = np.linalg.inv(A_a)
    beta_hat = A_inv @ b_a
    mu = float(x @ beta_hat)
    boost = float(np.sqrt(x @ A_inv @ x))
    return mu + c * boost, mu, boost

for t in range(T_ctx):
    x = rng2.standard_normal(d) * 0.5      # context for this user
    # Score each arm.
    scores = []
    for a in range(K):
        s, _, _ = linucb_score(x, A[a], b[a], c_ucb)
        scores.append(s)
    arm = int(np.argmax(scores))
    # True expected rewards this round.
    true_rewards = np.array([x @ beta_true[a] for a in range(K)])
    # Add noise and clip to [0, 1] so it behaves like a click probability.
    p = 0.5 + true_rewards     # center around 0.5
    p = np.clip(p, 0.01, 0.99)
    reward = int(rng2.random() < p[arm])
    # Regret is measured in expected click probability vs best arm.
    best_a = int(np.argmax(p))
    regret_t = p[best_a] - p[arm]
    # Update ridge regression state.
    A[arm] += np.outer(x, x)
    b[arm] += reward * x

    rewards[t] = reward
    regrets[t] = regret_t
    arm_picks[t] = arm
    for a in range(K):
        beta_hat_hist[t, a] = np.linalg.inv(A[a]) @ b[a]

print(f"Total clicks over {T_ctx} rounds: {int(rewards.sum())}")
print(f"Arm 0 picked {(arm_picks == 0).sum()} times, arm 1 picked {(arm_picks == 1).sum()} times.")
print(f"Total expected regret: {regrets.sum():.2f}")

In [ ]:
# Act 2 plots.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Plot 1: estimated beta_a[0] over time vs true value.
ts = np.arange(T_ctx)
axes[0].plot(ts, beta_hat_hist[:, 0, 0], color='#6366f1', lw=2, label='Arm 0, feature 0 (learned)')
axes[0].plot(ts, beta_hat_hist[:, 1, 0], color='#f59e0b', lw=2, label='Arm 1, feature 0 (learned)')
axes[0].axhline(beta_true[0, 0], color='#6366f1', ls='--', alpha=0.5, label='Arm 0 truth')
axes[0].axhline(beta_true[1, 0], color='#f59e0b', ls='--', alpha=0.5, label='Arm 1 truth')
axes[0].set_xlabel('round t')
axes[0].set_ylabel('estimated coefficient on feature 0')
axes[0].set_title('LinUCB learns both arms\' regressions')
axes[0].legend(loc='center right', fontsize=8)

# Plot 2: cumulative regret.
axes[1].plot(ts, np.cumsum(regrets), color='#10b981', lw=2)
axes[1].set_xlabel('round t')
axes[1].set_ylabel('cumulative regret (expected click gap)')
axes[1].set_title('Cumulative regret grows sublinearly')

plt.tight_layout()
plt.show()

print("LinUCB correctly learned that arm 0 has a large positive coefficient on feature 0,")
print("while arm 1 has a large negative one. That's enough to personalize: for users with")
print("positive feature 0, pick arm 0; for users with negative feature 0, pick arm 1.")

## Act 3: a real contextual bandit on real users

The toy example is fine. Now the real deal.

**Paper.** Offer-Westort, Rosenzweig & Athey (2024), *Battling the coronavirus 'infodemic' among social media users in Kenya and Nigeria*, *Nature Human Behaviour*.

**Subjects.** 15,292 Facebook Messenger users (7,498 Kenya, 7,794 Nigeria).

**Arms.** A 40-cell factorial of interventions designed to reduce intent-to-share misinformation: accuracy prompts, warning flags, related-article suggestions, etc.

**Algorithm.** The authors used **balanced Linear Thompson Sampling (BLTS)**, a version of LinTS with batched updates and inverse-propensity reweighting. Exactly the kind of thing we just built in Act 2, with two upgrades: Bayesian (posteriors over $\beta_a$) instead of UCB, and bias-corrected (IPW) to handle the batched adaptive assignment.

**Replication package.** All data and code are public at [gsbDBI/infodemic-replication](https://github.com/gsbDBI/infodemic-replication). We will load the primary tidy CSV directly from raw GitHub, no authentication needed.

**What we'll do.** Load the data. Show the structure. Collapse the 40 arms down to 4 interpretable ones. Run a small contextual bandit on the real context features and see whether we recover the paper's finding that accuracy nudges beat warning flags.

In [ ]:
# Load the Offer-Westort / Rosenzweig / Athey 2024 replication data.
# Source: github.com/gsbDBI/infodemic-replication (public, MIT-like).

df = pd.read_csv(DATA_URL)
print(f"Shape: {df.shape}  (rows x columns)")
print(f"Rows:  {len(df):,}   users from Kenya and Nigeria")
print(f"Cols:  {df.shape[1]}   variables per user")
print(f"Countries: {df['nigeria'].value_counts().to_dict()}  (1 = Nigeria, 0 = Kenya)")
print()
print("First few rows, key columns:")
display_cols = ['ID', 'W', 'Y', 'nigeria', 'male', 'age', 'ed', 'urban', 'batch']
existing = [c for c in display_cols if c in df.columns]
df[existing].head()

In [ ]:
# What arms were actually deployed, and to how many users?
print("All 40 arms, counts:")
print(df['W'].value_counts())
print()
print(f"Attrited users (not in paper's analysis): {int(df['attrited'].sum())}")
print(f"Analysis sample (attrited == 0): {int((df['attrited'] == 0).sum())}   "
      f"(matches paper's n = 15,292)")

### Replicating the paper's headline

> **Sign-convention heads-up.** The replication CSV's `Y` column uses the *opposite* sign convention from the paper. The paper's headline is **"accuracy nudges cut misinformation sharing by 4.9%"** (a *negative* effect on misinformation). In the CSV, `Y` is a composite index where *higher is better* (more sharing of true stuff, less sharing of fake stuff). So our t-test will come out **positive** for the accuracy-nudge group, which means the same thing the paper reports: accuracy nudges push people toward better behavior. If you are cross-checking our number against the paper's, remember to flip the sign.

The paper's headline effect: **accuracy nudges reduce misinformation sharing by 4.9%** (estimate -2.3pp, s.e. 1.0, p = 0.021).

A note on the outcome variable. In the replication CSV, the column `Y` is not the binary "shared misinformation" outcome you might expect. It's a composite response index that the authors use internally. From their replication script: `Y` weights sharing of **true** stimuli positively and sharing of **misinformation** stimuli negatively, so **higher `Y` is a better outcome** (more willingness to share true content, less willingness to share misinformation). A positive difference between the accuracy-nudge group and the control group is therefore the "good" direction.

The accuracy-nudge arms are the ones whose names contain `R_accuracy`. Let's split the sample and compute the difference in `Y`.

In [ ]:
# Simple replication: does the accuracy nudge raise the composite response Y?
analysis = df[df['attrited'] == 0].copy()
analysis['accuracy_nudge'] = analysis['W'].str.contains('R_accuracy').astype(int)

y1 = analysis.loc[analysis['accuracy_nudge'] == 1, 'Y']
y0 = analysis.loc[analysis['accuracy_nudge'] == 0, 'Y']

diff = y1.mean() - y0.mean()
se = np.sqrt(y1.var(ddof=1) / len(y1) + y0.var(ddof=1) / len(y0))
z = diff / se

print(f"Accuracy-nudge group:  n = {len(y1):>5,}   mean Y = {y1.mean():+.4f}")
print(f"Control/other group:   n = {len(y0):>5,}   mean Y = {y0.mean():+.4f}")
print(f"Difference (acc - ctl): {diff:+.4f}   s.e. = {se:.4f}   z = {z:.2f}")
print()
print("Positive diff = accuracy nudge raised the composite response Y, which means")
print("the accuracy-nudge group tilted toward sharing more true stimuli and less")
print("misinformation. Same qualitative finding as the paper's 4.9% reduction.")
print()
print("This naive t-test is NOT the paper's estimator. The paper uses doubly-robust")
print("AIPW scores with causal forests and inverse-propensity weighting to handle")
print("the adaptive assignment. We skip all that here and still recover the direction.")

### The personalization gap

A headline effect tells you what happens on average. A contextual bandit asks a different question: does the best arm **depend on who the user is**?

We'll do the simplest possible version. Pick a handful of user features, fit one ridge regression per arm (accuracy nudge vs not), and see whether the two regressions land on different coefficients. If they do, then a contextual policy has room to personalize.

In [ ]:
# Per-arm ridge regression on 8 user features.
feature_cols = ['nigeria', 'male', 'age', 'ed', 'urban', 'dli', 'science', 'religiosity']

# Standardize features so coefficients are comparable.
X_all = analysis[feature_cols].astype(float).fillna(analysis[feature_cols].mean()).values
X_all = (X_all - X_all.mean(axis=0)) / X_all.std(axis=0)
X_all = np.column_stack([np.ones(len(X_all)), X_all])   # add intercept

y = analysis['Y'].values
mask_acc = analysis['accuracy_nudge'].values == 1
mask_ctl = ~mask_acc

def fit_ridge(X, y, lam=1.0):
    A = X.T @ X + lam * np.eye(X.shape[1])
    b = X.T @ y
    return np.linalg.solve(A, b)

beta_acc = fit_ridge(X_all[mask_acc], y[mask_acc])
beta_ctl = fit_ridge(X_all[mask_ctl], y[mask_ctl])

labels = ['intercept'] + feature_cols
print(f"{'feature':<14}{'beta_accuracy':>15}{'beta_control':>15}{'diff':>15}")
for name, ba, bc in zip(labels, beta_acc, beta_ctl):
    print(f"{name:<14}{ba:>15.4f}{bc:>15.4f}{ba - bc:>15.4f}")
print()
print("Features where the two columns differ most are the features along which")
print("a contextual bandit would personalize.")

In [ ]:
# Plot the per-user treatment effect: how much does each user benefit from
# the accuracy nudge, according to the fitted regressions?

# Predicted Y under each arm, for every user.
y_hat_acc = X_all @ beta_acc
y_hat_ctl = X_all @ beta_ctl
# Positive uplift = accuracy nudge raises the composite response = good.
uplift = y_hat_acc - y_hat_ctl

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(uplift, bins=40, color='#6366f1', alpha=0.85, edgecolor='white')
ax.axvline(uplift.mean(), color='#f59e0b', lw=2, label=f'average = {uplift.mean():+.3f}')
ax.axvline(0, color='grey', ls='--')
ax.set_xlabel('predicted Y(accuracy) - Y(control)  (positive = accuracy nudge helps)')
ax.set_ylabel('users')
ax.set_title(f'Distribution of personalized treatment effects, n = {len(uplift):,}')
ax.legend()
plt.tight_layout()
plt.show()

n_benefit = (uplift > 0).sum()
print(f"Users for whom the fit predicts the accuracy nudge helps:   {n_benefit:,} ({100*n_benefit/len(uplift):.1f}%)")
print(f"Users for whom the fit predicts the accuracy nudge hurts:   {len(uplift)-n_benefit:,} ({100*(1-n_benefit/len(uplift)):.1f}%)")
print()
print("The point: a contextual policy would NOT give everyone the accuracy nudge.")
print("It would give it only to the users whose personalized uplift is positive.")
print("That's personalization through optimization, on real Facebook Messenger users.")

## Wrap up

Three acts, same machinery.

1. **Act 1** was classical bandits with no context. Thompson sampling converged on the better ad.
2. **Act 2** was contextual bandits on synthetic data. LinUCB learned two regressions, one per arm, and used them to personalize.
3. **Act 3** was the same contextual bandit idea on real Facebook Messenger users. We replicated the paper's headline and showed that a personalized policy would give different users different nudges.

### What to read next

- The paper itself: Offer-Westort, Rosenzweig & Athey (2024), *Nature Human Behaviour*. The methods section describes their balanced LinTS algorithm in detail.
- The replication repo: [gsbDBI/infodemic-replication](https://github.com/gsbDBI/infodemic-replication). The `code/` directory has R and Python scripts for the full analysis, including the propensity weighting and the restricted policy we skipped.
- Li, Chu, Langford & Schapire (2010). The original LinUCB paper, written for a Yahoo News deployment on 45 million page views. This is the ancestor of every contextual-bandit product running in industry today.

### What we skipped on purpose

- The 40-arm factorial structure. The paper's bandit allocates across all 40 cells using batched BLTS. We collapsed to 2 arms for pedagogy.
- The inverse-propensity weighting that corrects for batched adaptive assignment bias. If you're going to run your own adaptive experiment, read the paper's methods section and the replication scripts, not this notebook.
- The second stage of the paper (a confirmatory targeted-policy deployment on the learned optimal policy). The first stage is the one that teaches the contextual-bandit idea cleanly.

---

**Questions?** Bring them to Wednesday's class or drop them on Courseworks.